# NB1 — Data audit & index builder

**Purpose.** Verify the Sen1Floods11 mirror matches every assumption the `lgmf`
module makes, then emit the artifacts the other notebooks load:

- `train_index.json` / `val_index.json` / `test_index.json` — paired paths on the **official** split
- `bolivia_index.json` — the out-of-region Bolivia holdout (a clean generalization test)
- `norm_stats.json` — per-band mean/std (computed on the **train split only**)

**Attach:** the `lgmf_c` code package and the Sen1Floods11 dataset. The official
split CSVs are fetched automatically — either from an attached dataset if you
have one, or downloaded over HTTPS (needs **Internet: On** in notebook settings).
Run top to bottom, then *Save Version* and turn `/kaggle/working` into a Dataset
named `lgmf-index`.


In [ ]:
# --- bootstrap: put the lgmf package on the path ---
import sys, os, glob, json, csv, urllib.request, warnings
warnings.filterwarnings("ignore")

init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "lgmf package not found -- attach the lgmf_c dataset"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))

import numpy as np
import rasterio
import lgmf
from lgmf.config import DEFAULT_S2_BANDS
print("lgmf", lgmf.__version__, "| S2 band subset:", DEFAULT_S2_BANDS)

## 1. Locate the HandLabeled folders

In [ ]:
label_files = glob.glob("/kaggle/input/**/LabelHand/*.tif", recursive=True)
assert label_files, "No LabelHand tifs found -- is the Sen1Floods11 dataset attached?"
HAND_DIR = os.path.dirname(os.path.dirname(label_files[0]))   # .../HandLabeled
print("HandLabeled root:", HAND_DIR)
for sub in ["S1Hand", "S2Hand", "LabelHand", "JRCWaterHand", "S1OtsuLabelHand"]:
    p = os.path.join(HAND_DIR, sub)
    n = len(glob.glob(p + "/*.tif")) if os.path.isdir(p) else 0
    print(f"  {sub:16s} {'OK ' if os.path.isdir(p) else 'MISSING'}  {n} files")

## 2. Pair chips by filename prefix

In [ ]:
def chip_id(fname):
    # 'Bolivia_103757_LabelHand.tif' -> 'Bolivia_103757'
    return os.path.basename(fname).rsplit("_", 1)[0]

def find(sub, cid, suffix):
    p = os.path.join(HAND_DIR, sub, f"{cid}_{suffix}.tif")
    return p if os.path.exists(p) else None

ids = sorted({chip_id(f) for f in label_files})
records, missing = [], []
for cid in ids:
    r = {"id": cid,
         "label": find("LabelHand", cid, "LabelHand"),
         "s1": find("S1Hand", cid, "S1Hand"),
         "s2": find("S2Hand", cid, "S2Hand"),
         "jrc": find("JRCWaterHand", cid, "JRCWaterHand")}
    if r["label"] and r["s1"] and r["s2"]:
        records.append(r)
    else:
        missing.append(cid)

print(f"{len(records)} fully-paired chips | {len(missing)} incomplete")
if missing[:5]:
    print("  incomplete examples:", missing[:5])
have_jrc = sum(r["jrc"] is not None for r in records)
print(f"  chips with JRC permanent-water: {have_jrc}/{len(records)}")

## 3. Sanity checks — band counts, size, label values

In [ ]:
ex = records[0]
with rasterio.open(ex["s1"]) as s: s1_bands = s.count
with rasterio.open(ex["s2"]) as s: s2_bands, H, W = s.count, s.height, s.width
with rasterio.open(ex["label"]) as s: lbl0 = s.read(1)

print(f"S1 bands = {s1_bands} (expect 2)")
print(f"S2 bands = {s2_bands} (expect 13)")
print(f"chip size = {H} x {W}")
assert s1_bands == 2, "S1 should have VV,VH"
if s2_bands != 13:
    print(f"  !! S2 has {s2_bands} bands, not 13 -- DEFAULT_S2_BANDS indices may be wrong; verify order")

print("label unique values:", np.unique(lbl0), "(expect subset of {-1,0,1})")

frac = []
for r in records[::10]:
    with rasterio.open(r["label"]) as s: l = s.read(1)
    frac.append(float((l == -1).mean()))
print(f"label==-1 (no-data/cloud): mean {np.mean(frac):.3f}, max {np.max(frac):.3f}")

## 4. Verify S2 **band order** with an MNDWI check

The module assumes subset order `[B2,B3,B4,B8,B11,B12]` -> green is index 1,
SWIR-1 is index 4. Water should have a **higher** MNDWI than land. If this
fails, the band indices are wrong -- fix `DEFAULT_S2_BANDS` in `config.py`.


In [ ]:
S2B_1 = [b + 1 for b in DEFAULT_S2_BANDS]   # rasterio is 1-based
g_i, sw_i = 1, 4                            # green, swir1 in subset order
w_vals, l_vals = [], []
for r in records[:40]:
    with rasterio.open(r["s2"]) as s: arr = s.read(S2B_1).astype("float32")
    with rasterio.open(r["label"]) as s: l = s.read(1)
    g, sw = arr[g_i], arr[sw_i]
    mndwi = (g - sw) / (g + sw + 1e-6)
    if (l == 1).any(): w_vals.append(mndwi[l == 1].mean())
    if (l == 0).any(): l_vals.append(mndwi[l == 0].mean())
mw, ml = np.mean(w_vals), np.mean(l_vals)
print(f"MNDWI  water={mw:+.3f}  land={ml:+.3f}")
print("PASS: band order looks correct (water > land)" if mw > ml
      else "FAIL: water MNDWI not higher -- fix DEFAULT_S2_BANDS")

## 5. Official train / val / test / bolivia split

Uses the **official** Sen1Floods11 hand-labeled split CSVs for comparability with
SMAGNet / DMCF-Net. The CSVs are found in one of three ways, in order:

1. An **attached dataset** containing `flood_*_data.csv` (best: fully reproducible, no network).
2. **HTTPS download** from the public GCS bucket (needs Internet: On).
3. Loud **deterministic fallback** if neither works.

Expected: train ~252, valid ~89, test ~90, bolivia ~15 (the Bolivia holdout is
kept **separate** as an out-of-region test, not folded into train).


In [ ]:
SPLIT_FILES = {"train": "flood_train_data.csv", "valid": "flood_valid_data.csv",
               "test": "flood_test_data.csv", "bolivia": "flood_bolivia_data.csv"}
GCS_BASE = "https://storage.googleapis.com/sen1floods11/v1.1/splits/flood_handlabeled"

def ids_from_csv(path):
    out = set()
    with open(path) as f:
        for row in csv.reader(f):
            for cell in row:
                cell = cell.strip()
                if cell.endswith(".tif"):
                    out.add(chip_id(cell)); break
    return out

def locate_official():
    # 1) attached dataset
    found = {}
    for key, fn in SPLIT_FILES.items():
        c = glob.glob(f"/kaggle/input/**/{fn}", recursive=True)
        if c:
            found[key] = c[0]
    if "train" in found and "test" in found:
        return found, "attached"
    # 2) HTTPS download
    os.makedirs("/kaggle/working/splits", exist_ok=True)
    dl = {}
    for key, fn in SPLIT_FILES.items():
        dst = f"/kaggle/working/splits/{fn}"
        try:
            urllib.request.urlretrieve(f"{GCS_BASE}/{fn}", dst)
            dl[key] = dst
        except Exception as e:
            print(f"  download failed for {fn}: {e}")
    if "train" in dl and "test" in dl:
        return dl, "downloaded"
    return {}, "none"

paths, origin = locate_official()
by_id = {r["id"]: r for r in records}
pick = lambda p: [by_id[i] for i in ids_from_csv(p) if i in by_id] if p else []

if paths:
    train = pick(paths.get("train")); val = pick(paths.get("valid"))
    test = pick(paths.get("test")); bolivia = pick(paths.get("bolivia"))
    SPLIT_SOURCE = f"official ({origin})"
else:
    import random
    print("Could NOT obtain official splits (no attached CSVs and no internet).")
    print("  >>> Falling back to a deterministic 60/20/20 split -- NOT comparable")
    print("      to prior work. Attach the split CSVs or enable Internet, then re-run. <<<")
    rng = random.Random(42); pool = records[:]; rng.shuffle(pool)
    n = len(pool); a, b = int(0.6 * n), int(0.8 * n)
    train, val, test, bolivia = pool[:a], pool[a:b], pool[b:], []
    SPLIT_SOURCE = "fallback 60/20/20"

print(f"split = {SPLIT_SOURCE}")
print(f"  train {len(train)}  val {len(val)}  test {len(test)}  bolivia {len(bolivia)}")
assert train and test, "empty split -- check CSV parsing or fallback"

## 5b. Drop fully-masked chips — **train only**

A chip that is ~all no-data (`label == -1`) contributes zero gradient and can
destabilize BatchNorm in small batches. We remove those from **train only**;
val / test / bolivia are left exactly as the official split defines them, so the
benchmark comparison stays honest.


In [ ]:
DROP_THRESH = 0.98
def masked_frac(r):
    with rasterio.open(r["label"]) as s: l = s.read(1)
    return float((l == -1).mean())

before = len(train)
dropped = [r["id"] for r in train if masked_frac(r) > DROP_THRESH]
train = [r for r in train if r["id"] not in dropped]
print(f"train fully-masked filter (> {DROP_THRESH:.0%} no-data): "
      f"dropped {len(dropped)}/{before}")
if dropped:
    print("  dropped:", dropped[:10])
print(f"train after filter: {len(train)}")

## 6. Write the index files

In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)
keys = ("id", "s1", "s2", "label", "jrc")
def clean(recs): return [{k: r[k] for k in keys} for r in recs]
for name, recs in [("train", train), ("val", val),
                   ("test", test), ("bolivia", bolivia)]:
    if not recs:
        print(f"(skip {name}_index.json -- empty)")
        continue
    with open(f"/kaggle/working/{name}_index.json", "w") as f:
        json.dump(clean(recs), f, indent=1)
    print(f"wrote {name}_index.json  ({len(recs)} chips)")

## 7. Normalization stats (train split only)

Computed on **raw** pixel values so they plug straight into `cfg.s1_mean/std`
and `cfg.s2_mean/std`. Pixels subsampled 4x for speed; S1 zero-fill and
non-finite pixels excluded.


In [ ]:
def compute_stats(recs, which, bands_1based=None, subsample=4):
    sums = sqs = cnt = None
    for r in recs:
        with rasterio.open(r[which]) as s:
            arr = (s.read(bands_1based) if bands_1based else s.read()).astype("float64")
        arr = arr[:, ::subsample, ::subsample]
        C = arr.shape[0]
        flat = arr.reshape(C, -1)
        mask = np.isfinite(flat)
        if which == "s1":
            mask &= (flat != 0)
        if sums is None:
            sums = np.zeros(C); sqs = np.zeros(C); cnt = np.zeros(C)
        for c in range(C):
            v = flat[c][mask[c]]
            sums[c] += v.sum(); sqs[c] += (v * v).sum(); cnt[c] += v.size
    mean = sums / np.maximum(cnt, 1)
    std = np.sqrt(np.maximum(sqs / np.maximum(cnt, 1) - mean ** 2, 1e-12))
    return mean.tolist(), std.tolist()

s1_mean, s1_std = compute_stats(train, "s1")
s2_mean, s2_std = compute_stats(train, "s2", bands_1based=S2B_1)
print("S1 mean", np.round(s1_mean, 3), "std", np.round(s1_std, 3))
print("S2 mean", np.round(s2_mean, 3), "std", np.round(s2_std, 3))

stats = {"s1_mean": s1_mean, "s1_std": s1_std,
         "s2_mean": s2_mean, "s2_std": s2_std,
         "s2_bands": list(DEFAULT_S2_BANDS), "split_source": SPLIT_SOURCE,
         "dropped_train_chips": dropped}
with open("/kaggle/working/norm_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print("wrote norm_stats.json")

## 8. Class balance — water, and permanent vs flood (JRC)

In [ ]:
tot_valid = tot_water = tot_perm = 0
for r in train:
    with rasterio.open(r["label"]) as s: l = s.read(1)
    valid = (l != -1); water = (l == 1) & valid
    tot_valid += int(valid.sum()); tot_water += int(water.sum())
    if r["jrc"]:
        with rasterio.open(r["jrc"]) as s: j = s.read(1) > 0
        tot_perm += int((water & j).sum())
print(f"water = {tot_water / max(tot_valid,1):.3%} of valid pixels")
if tot_perm:
    perm_frac = tot_perm / max(tot_water, 1)
    print(f"of water: permanent {perm_frac:.1%} | flood {1 - perm_frac:.1%}")
    print("  -> report flood-only IoU (JRC-excluded) as the flood-specific metric")

## 9. Figure 2 — dataset overview triplets (S1 | S2 | label)

In [ ]:
from lgmf.viz import plot_triplet
os.makedirs("/kaggle/working/figs", exist_ok=True)
for k, r in enumerate(train[:3]):
    with rasterio.open(r["s1"]) as s: s1 = s.read().astype("float32")
    with rasterio.open(r["s2"]) as s: s2 = s.read(S2B_1).astype("float32")
    with rasterio.open(r["label"]) as s:
        l = s.read(1).astype("float32"); l[l < 0] = 0
    jrc = None
    if r["jrc"]:
        with rasterio.open(r["jrc"]) as s: jrc = (s.read(1) > 0).astype("float32")
    plot_triplet(s1, s2, l, f"/kaggle/working/figs/fig2_triplet_{k}", jrc=jrc)
print("saved triplet figures to /kaggle/working/figs")

## 10. Summary & next step

In [ ]:
print("=" * 52)
print("NB1 outputs in /kaggle/working:")
for f in ["train_index.json", "val_index.json", "test_index.json",
          "bolivia_index.json", "norm_stats.json"]:
    ok = os.path.exists("/kaggle/working/" + f)
    print(f"  {'OK ' if ok else '-- '} {f}")
print(f"\nsplit source: {SPLIT_SOURCE}")
if not SPLIT_SOURCE.startswith("official"):
    print("  WARNING: not on the official split -- results won't be comparable.")
print("\nNEXT:")
print("  1. Save Version (commit) so /kaggle/working persists.")
print("  2. Create a Kaggle Dataset from these outputs, named 'lgmf-index'.")
print("  3. In NB2: attach lgmf-index, then")
print("       cfg.split_dir = '/kaggle/input/lgmf-index'")
print("       s = json.load(open(cfg.split_dir + '/norm_stats.json'))")
print("       cfg.s1_mean, cfg.s1_std = s['s1_mean'], s['s1_std']")
print("       cfg.s2_mean, cfg.s2_std = s['s2_mean'], s['s2_std']")